In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import urllib3
import re

# SSL 인증서 경고 비활성화
# 일부 환경에서 HTTPS 인증서 경고가 발생할 수 있으므로
# 실습 중 불필요한 경고 메시지를 숨긴다.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# 문제 1. 웹페이지 가져오기
# 서울시 교통정보 페이지의 HTML을 요청한다.
# verify=False를 사용하여 SSL 인증서 검증을 비활성화한다.
url = "https://news.seoul.go.kr/traffic/archives/1551"
response = requests.get(url, verify=False)

print("문제 1. 웹페이지 가져오기")
print("답: 상태 코드:", response.status_code)

문제 1. 웹페이지 가져오기
답: 상태 코드: 200


In [3]:
# 문제 2. HTML 파싱 및 첫 번째 테이블 선택
# BeautifulSoup으로 HTML을 파싱한 뒤, 페이지 안의 모든 <table> 태그를 찾는다.
# 이후 첫 번째 테이블만 사용한다.
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.select("table")

print("\n문제 2. 첫 번째 테이블 찾기")
print(f"답: 전체 테이블 수: {len(tables)}")

if not tables:
    raise ValueError("페이지에서 테이블을 찾지 못했습니다.")

# 첫 번째 테이블 선택
first_table = tables[0]


문제 2. 첫 번째 테이블 찾기
답: 전체 테이블 수: 3


In [4]:
# 문제 3. HTML 테이블 구조 펼치기 함수 작성
# HTML 표에는 rowspan, colspan 같은 병합 셀이 있을 수 있으므로,
# 이를 고려하여 실제 2차원 표 형태(list of list)로 변환한다.
def expand_html_table(table):
    """
        HTML에서는 다음과 같은 구조가 가능하다
        - 하나의 셀이 여러 열을 차지 (colspan)
        - 하나의 셀이 여러 행에 걸쳐 있음 (rowspan)

    하지만 단순히 <td> 텍스트만 추출하면
        - 열 개수가 맞지 않거나
        - 데이터가 밀리는 문제가 발생한다

    따라서:
        -> 병합 셀을 실제로 펼쳐서 모든 행이 동일한 길이를 가지도록 만든다
    """
    rows = table.find_all("tr")

    # rowspan 때문에 아래 행까지 채워야 하는 값을 저장
    # key: 열 인덱스, value: [남은 rowspan 횟수, 텍스트]
    span_map = {}
    grid = []

    for tr in rows:
        current_row = []
        cells = tr.find_all(["th", "td"])

        cell_idx = 0
        col_idx = 0

        while cell_idx < len(cells) or col_idx in span_map:

            # 이전 행의 rowspan 값이 남아 있으면 먼저 채운다.
            if col_idx in span_map:
                remain, text = span_map[col_idx]
                current_row.append(text)

                if remain > 1:
                    span_map[col_idx] = [remain - 1, text]
                else:
                    del span_map[col_idx]

                col_idx += 1
                continue

            # 현재 셀의 텍스트 추출
            cell = cells[cell_idx]
            text = re.sub(r"\s+", " ", cell.get_text(" ", strip=True)).strip()

            # rowspan, colspan 정보 읽기
            rowspan = int(cell.get("rowspan", 1))
            colspan = int(cell.get("colspan", 1))

            # colspan 만큼 가로 방향으로 값 채우기
            for offset in range(colspan):
                current_row.append(text)

                # rowspan이 있으면 이후 행에도 같은 값을 유지
                if rowspan > 1:
                    span_map[col_idx + offset] = [rowspan - 1, text]

            col_idx += colspan
            cell_idx += 1

        grid.append(current_row)

    # 모든 행 길이를 동일하게 맞춤
    max_cols = max(len(row) for row in grid)
    grid = [row + [""] * (max_cols - len(row)) for row in grid]

    return grid

In [5]:
# 문제 4. 중복 컬럼명 처리 함수 작성
# pandas DataFrame에서 같은 이름의 컬럼이 여러 개 있으면
# 이후 분석 시 혼란이 생길 수 있으므로 __2, __3 형태로 구분한다.
def make_unique_columns(columns):
    """
    pandas DataFrame에서 컬럼명이 중복되면 혼란이 생기므로 __2, __3 형태로 고유하게 만든다.
    """
    seen = {}
    result = []

    for col in columns:
        col = re.sub(r"\s+", " ", str(col)).strip() or "col"

        if col not in seen:
            seen[col] = 1
            result.append(col)
        else:
            seen[col] += 1
            result.append(f"{col}__{seen[col]}")

    return result

In [6]:
# 문제 5. 첫 번째 테이블 펼치기
# expand_html_table 함수를 사용하여
# 병합 셀 구조를 일반적인 2차원 리스트 형태로 변환한다.
data = expand_html_table(first_table)

print("\n문제 5. 펼친 테이블 일부 확인")
for row in data[:5]:
    print(row)


문제 5. 펼친 테이블 일부 확인
['구 분', '계', '1호선', '2 호 선', '2 호 선', '2 호 선', '3호선', '4호선', '5호선', '6호선', '7호선', '8호선', '9호선', '우이 신설선', '신림선']
['구 분', '계', '1호선', '순 환', '지 선', '지 선', '3호선', '4호선', '5호선', '6호선', '7호선', '8호선', '9호선', '우이 신설선', '신림선']
['구 간', '11개', '서울역 ~ 청량리', '성수 ~ 성수', '신설동 ~ 성수', '신도림 ~ 까치산', '지축 ~ 오금', '당고개 ~ 남태령', '방화 ~ 하남 검단산 /마천', '응암 ~ 신내', '장암 ~ 온수', '암사 역사 공원 ~ 모란', '개화 ~ 중앙 보훈 병원', '북한산 우이 ~ 신설동', '샛강 ~ 관악산']
['영업거리 (km)', '359. 86', '7.8', '48.8', '5.4', '6', '38.2', '31.7', '59.8', '36.3', '46.9', '19.2', '40.6', '11.4', '7.76']
['역 수', '338', '10', '43', '4', '3', '34', '26', '56', '39', '42', '19', '38', '13', '11']


In [9]:
# 문제 6. 2단 헤더 결합
# 첫 번째 표는 상위 헤더 + 하위 헤더 구조를 가진다.
# 예:
#   상위 헤더: 2호선
#   하위 헤더: 순환 / 지선
# 따라서 두 행을 결합하여 최종 컬럼명을 만든다.
header1 = data[0]
header2 = data[1]

columns = []

for a, b in zip(header1, header2):
    a = re.sub(r"\s+", " ", str(a)).strip()
    b = re.sub(r"\s+", " ", str(b)).strip()

    # 두 값이 같으면 하나만 사용
    if a == b:
        col = a

    # 상위 헤더만 있고 하위 헤더가 비어 있으면 상위 헤더 사용
    elif a and not b:
        col = a

    # 하위 헤더만 있고 상위 헤더가 비어 있으면 하위 헤더 사용
    elif not a and b:
        col = b

    # 둘 다 값이 있으면 상위 + 하위 형태로 결합
    else:
        # 구분, 계는 중복 결합하지 않도록 처리
        if a in ["구분", "구 분", "계"]:
            col = a
        else:
            col = f"{a} {b}"

    columns.append(col)

# 중복 컬럼명 정리
columns = make_unique_columns(columns)

# 실제 데이터는 헤더 2줄 이후부터 사용
body = data[2:]
df = pd.DataFrame(body, columns=columns)

In [10]:
# 문제 7. 결과 확인
# 컬럼명이 의도대로 생성되었는지 확인한다.
# 예:
#   2호선 순환
#   2호선 지선
print("\n문제 7. 컬럼 구조 확인")
print(df.columns.tolist())


문제 7. 컬럼 구조 확인
['구 분', '계', '1호선', '2 호 선 순 환', '2 호 선 지 선', '2 호 선 지 선__2', '3호선', '4호선', '5호선', '6호선', '7호선', '8호선', '9호선', '우이 신설선', '신림선']


In [11]:
print("\n 상위 데이터 확인")
df.head()


 상위 데이터 확인


,구 분,계,1호선,2 호 선 순 환,2 호 선 지 선,2 호 선 지 선__2,3호선,4호선,5호선,6호선,7호선,8호선,9호선,우이 신설선,신림선
0,구 간,11개,서울역 ~ 청량리,성수 ~ 성수,신설동 ~ 성수,신도림 ~ 까치산,지축 ~ 오금,당고개 ~ 남태령,방화 ~ 하남 검단산 /마천,응암 ~ 신내,장암 ~ 온수,암사 역사 공원 ~ 모란,개화 ~ 중앙 보훈 병원,북한산 우이 ~ 신설동,샛강 ~ 관악산
1,영업거리 (km),359. 86,7.8,48.8,5.4,6,38.2,31.7,59.8,36.3,46.9,19.2,40.6,11.4,7.76
2,역 수,338,10,43,4,3,34,26,56,39,42,19,38,13,11
3,소요시간 (분),9 ~ 109.5,18,90,9,11,67.5,53,109.5,73.8,87,51.5,79.8 (일반) 52.5 (급행),23,18.1
4,운행시격 (분),RH,3.0,2.5,7.0,10.0,3.0,2.5,2.5,4.0,2.5,4.5,3.1,2.9,4
